In [ ]:
# Cell 1
import os, sys
sys.path.append(os.path.abspath(".."))
import tensorflow as tf
import numpy as np
from sklearn.metrics import f1_score, average_precision_score
from src import config
from src.data_pipeline import load_merged_dataframe, get_column_groups, split_partitions, normalize_targets, make_dataset

df = load_merged_dataframe()
attr_cols, bbox_cols, landmark_cols = get_column_groups(df)
_, _, test_df = split_partitions(df)
test_df = normalize_targets(test_df, bbox_cols, landmark_cols)
test_ds = make_dataset(test_df, attr_cols, bbox_cols, landmark_cols)

model = tf.keras.models.load_model(os.path.join(config.DATA_DIR, "multitask_face_model.keras"))

In [ ]:
# Cell 2 — same metrics as the original cell 38
results = model.evaluate(test_ds, return_dict=True)
print(results)

attr_pred, land_pred, box_pred = model.predict(test_ds)
attr_true = (test_df[attr_cols].values == 1).astype(int)
attr_pred_labels = (attr_pred > 0.5).astype(int)

print(f"Attributes macro-F1: {f1_score(attr_true, attr_pred_labels, average='macro'):.4f}")
print(f"Attributes mAP:      {average_precision_score(attr_true, attr_pred, average='macro'):.4f}")
print(f"Landmark MAE (normalized): {np.abs(land_pred - test_df[landmark_cols].values).mean():.4f}")
print(f"Bbox MAE (normalized):     {np.abs(box_pred - test_df[bbox_cols].values).mean():.4f}")